# Milan Airbnb and Rental Dynamics

## 0) Brief Project Description - Import Libraries and Data

This notebook reorganizes the project into one transparent analytical narrative. The core question is whether the current geography of Airbnb activity in Milan helps us interpret the evolution of long term rents recorded by the official OMI datasets. The intended audience is a non expert reader who needs a clear path from raw files to interpretable evidence, not only a gallery of charts.

The workflow begins with three raw sources. The Airbnb listings file describes the current market structure, the reviews file provides a real temporal signal of platform activity, and the semester based OMI files describe the rental market by zone from 2018 to 2024. We start by importing the data and establishing a shared visual language that uses the Milan palette chosen for the project.

In [ ]:
# Import the libraries used across the notebook.
from pathlib import Path
import importlib.util
import warnings

# Import the numerical and tabular stack.
import numpy as np
import pandas as pd

# Import the visualization stack.
import matplotlib.pyplot as plt
import seaborn as sns

# Hide uninformative warnings during the exploratory workflow.
warnings.filterwarnings("ignore")

# Define the notebook and project paths.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "src" else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"

# Define the generated output folders requested for the assignment.
IO_DIR = SRC_DIR / "io"
CLEAN_DIR = IO_DIR / "cleaned_data"
PLOS_DIR = IO_DIR / "plos"

# Define the public output folder for reports and exported plots.

# Create the folders before generating any file.
for folder in [IO_DIR, CLEAN_DIR, PLOS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Store the Milan palette in a reusable dictionary.
palette = {
    "coral": "#FF5A5F",
    "white": "#FFFFFF",
    "duomo_pink": "#E8B4B8",
    "red": "#D2232A",
    "gold": "#F4B82D",
    "green": "#1A5E3F",
    "terracotta": "#8B3A2F",
}

# Apply a clean plotting theme for the whole notebook.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.facecolor"] = palette["white"]
plt.rcParams["figure.facecolor"] = palette["white"]
plt.rcParams["axes.edgecolor"] = "#D9D9D9"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"] = 10

# Load the raw Airbnb listings and reviews files.
listings_raw = pd.read_csv(DATA_DIR / "listings_milan.csv")
reviews_raw = pd.read_csv(DATA_DIR / "reviews_milan.csv")

# Collect the raw OMI semester files in chronological order.
omi_files = sorted(DATA_DIR.glob("quotazioni_omi_locazione_*.csv"))

# Summarize the imported sources before cleaning them.
dataset_inventory = pd.DataFrame(
    [
        {"dataset": "Airbnb listings raw", "rows": len(listings_raw), "columns": listings_raw.shape[1]},
        {"dataset": "Airbnb reviews raw", "rows": len(reviews_raw), "columns": reviews_raw.shape[1]},
        {"dataset": "OMI semester files", "rows": len(omi_files), "columns": 18},
    ]
)

# Display the imported structure and a small preview of the inputs.
display(dataset_inventory)
display(listings_raw.head(3))
display(reviews_raw.head(3))
print("First OMI files:", [path.name for path in omi_files[:3]])
print("Last OMI files:", [path.name for path in omi_files[-3:]])

The import step already clarifies the logic of the project. The listings table is a snapshot that describes the present structure of supply, while the reviews table is much larger and is better suited for temporal analysis. The OMI material is fragmented across fourteen semester files, which means that integration is not optional but central to the project itself.

This distinction matters analytically. If we confuse a current snapshot with a genuine time series, we risk telling an attractive but unsupported story. For that reason the preprocessing stage keeps the snapshot logic of listings separate from the temporal logic of reviews and rents, and only later reconnects them through carefully defined aggregate tables.

## 1) Preprocessing

The preprocessing stage has two goals. The first goal is technical, because the raw files come in different formats, naming conventions, and levels of completeness. The second goal is interpretive, because the derived variables should already move the data closer to the questions we want to ask, such as how commercialized the Airbnb stock is, which listings behave like long term rentals, and how rents evolve across official market zones.

In [ ]:
# Copy the raw listings table to preserve the original import.
listings_clean = listings_raw.copy()

# Remove the empty group field that adds no information for Milan.
listings_clean = listings_clean.drop(columns=["neighbourhood_group"], errors="ignore")

# Standardize text fields used later in joins and plots.
for col in ["name", "host_name", "neighbourhood", "room_type", "license"]:
    listings_clean[col] = listings_clean[col].astype("string").str.strip()

# Normalize the neighborhood names to the style used by the mapping table.
listings_clean["neighbourhood"] = listings_clean["neighbourhood"].str.title()

# Parse the listing review date to a proper datetime field.
listings_clean["last_review"] = pd.to_datetime(listings_clean["last_review"], errors="coerce")

# Convert the numerical columns to a robust numeric dtype.
numeric_cols = ["price", "minimum_nights", "number_of_reviews", "reviews_per_month", "calculated_host_listings_count", "availability_365", "number_of_reviews_ltm", "latitude", "longitude"]
for col in numeric_cols:
    listings_clean[col] = pd.to_numeric(listings_clean[col], errors="coerce")

# Remove duplicated listing identifiers to keep one row per listing.
listings_clean = listings_clean.drop_duplicates(subset="id").copy()

# Derive the core interpretation fields for later sections.
listings_clean["is_entire_home"] = listings_clean["room_type"].eq("Entire home/apt")
listings_clean["is_professional_host"] = listings_clean["calculated_host_listings_count"].ge(5)
listings_clean["is_long_term"] = listings_clean["minimum_nights"].ge(30)

# Derive activity and licensing signals from the snapshot.
listings_clean["is_active"] = listings_clean["number_of_reviews_ltm"].fillna(0).gt(0)
listings_clean["has_license"] = listings_clean["license"].notna()
listings_clean["host_tier"] = np.select(
    [
        listings_clean["calculated_host_listings_count"].eq(1),
        listings_clean["calculated_host_listings_count"].between(2, 5, inclusive="both"),
        listings_clean["calculated_host_listings_count"].between(6, 20, inclusive="both"),
        listings_clean["calculated_host_listings_count"].ge(21),
    ],
    ["1 listing", "2-5 listings", "6-20 listings", "21+ listings"],
    default="Unknown",
)

# Derive a simple tier for minimum stay requirements.
listings_clean["nights_tier"] = np.select(
    [
        listings_clean["minimum_nights"].between(1, 2, inclusive="both"),
        listings_clean["minimum_nights"].between(3, 6, inclusive="both"),
        listings_clean["minimum_nights"].between(7, 29, inclusive="both"),
        listings_clean["minimum_nights"].ge(30),
    ],
    ["1-2 nights", "3-6 nights", "7-29 nights", "30+ nights"],
    default="Unknown",
)

# Distinguish formal codes from textual notes and missing licenses.
listings_clean["lic_category"] = np.select(
    [
        listings_clean["license"].isna(),
        listings_clean["license"].str.startswith("IT", na=False),
        listings_clean["license"].notna(),
    ],
    ["Missing", "Declared code", "Text note"],
    default="Missing",
)

# Save the cleaned listings table for the rest of the project.
listings_clean_path = CLEAN_DIR / "listings_milan_cleaned.csv"
listings_clean.to_csv(listings_clean_path, index=False)

# Build a compact quality summary for the cleaned listings file.
listings_quality = pd.DataFrame(
    {
        "metric": ["Rows", "Columns", "Missing prices", "Missing licenses", "Entire homes share", "Professional host share"],
        "value": [
            f"{len(listings_clean):,}",
            listings_clean.shape[1],
            int(listings_clean["price"].isna().sum()),
            int(listings_clean["license"].isna().sum()),
            f"{listings_clean['is_entire_home'].mean():.1%}",
            f"{listings_clean['is_professional_host'].mean():.1%}",
        ],
    }
)

# Display the key quality indicators and a small preview.
display(listings_quality)
display(listings_clean.head(3))
print("Saved:", listings_clean_path)

Cleaning the listings file does more than make the table prettier. The derived variables expose the structure of the market. In particular, the share of entire homes and the share of multi listing hosts indicate that a large part of the Milan offer is not a marginal form of home sharing, but a more commercialized segment of the housing stock. That insight becomes important later, because the project is interested in pressure on ordinary residential use.

It is also useful that we keep missing prices and licenses visible instead of forcing artificial fixes. Those gaps are not noise to hide. They tell us where the platform snapshot is incomplete, and they remind us that the cleaned file is an analytical convenience, not a claim that the underlying market is perfectly observed.

In [ ]:
# Copy the raw reviews table before transforming it.
reviews_clean = reviews_raw.copy()

# Parse the review date so that the file becomes a valid time series input.
reviews_clean["date"] = pd.to_datetime(reviews_clean["date"], errors="coerce")

# Remove exact duplicates and rows without a valid date.
reviews_clean = reviews_clean.drop_duplicates().dropna(subset=["date"]).copy()

# Build explicit calendar fields for later aggregations.
reviews_clean["year"] = reviews_clean["date"].dt.year
reviews_clean["month"] = reviews_clean["date"].dt.month
reviews_clean["year_month"] = reviews_clean["date"].dt.to_period("M").astype(str)

# Save the cleaned reviews file to the notebook output area.
reviews_clean_path = CLEAN_DIR / "reviews_milan_cleaned.csv"
reviews_clean.to_csv(reviews_clean_path, index=False)

# Summarize the temporal coverage of the review history.
reviews_quality = pd.DataFrame(
    {
        "metric": ["Rows", "Columns", "Unique listings", "First review", "Last review", "Covered years"],
        "value": [
            f"{len(reviews_clean):,}",
            reviews_clean.shape[1],
            f"{reviews_clean['listing_id'].nunique():,}",
            str(reviews_clean["date"].min().date()),
            str(reviews_clean["date"].max().date()),
            f"{reviews_clean['year'].min()} to {reviews_clean['year'].max()}",
        ],
    }
)

# Display the temporal quality report and a small preview.
display(reviews_quality)
display(reviews_clean.head(3))
print("Saved:", reviews_clean_path)

The reviews file is the real backbone of the temporal analysis. It stretches from late 2010 to late 2025 and links almost one million observations to individual listings. This does not mean that reviews measure the whole platform perfectly, because unreviewed stays remain invisible, but it does mean that we have a much more defensible time signal than anything we could infer from the listing snapshot alone.

A second implication is substantive. Because review activity extends through the pandemic shock and the subsequent recovery, the dataset allows us to observe a structural break in platform use. That makes the temporal story more credible and gives the later visualizations a concrete event around which trends can be interpreted.

In [ ]:
# Read and harmonize every OMI semester file.
omi_parts = []
for path in omi_files:
    year, semester = path.stem.replace("quotazioni_omi_locazione_", "").split("_")
    header_row = 1 if path.name == "quotazioni_omi_locazione_2023_1.csv" else 0
    part = pd.read_csv(path, sep=";", encoding="latin1", header=header_row)
    part["Year"] = int(year)
    part["Semester"] = int(semester)
    omi_parts.append(part)

# Concatenate the semester files into one long table.
omi_clean = pd.concat(omi_parts, ignore_index=True)

# Standardize text columns before any grouping step.
text_cols = ["Area_territoriale", "Regione", "Prov", "Comune_amm", "Comune_descrizione", "Fascia", "Zona", "LinkZona", "Cod_Tip", "Descr_Tipologia", "Stato", "Stato_prev", "Sup_NL_loc"]
for col in text_cols:
    omi_clean[col] = omi_clean[col].astype("string").str.strip()

# Keep the Milan rows even if future files include additional municipalities.
omi_clean = omi_clean[omi_clean["Comune_descrizione"].eq("MILANO")].copy()

# Convert the rent columns from decimal comma to numeric values.
for col in ["Loc_min", "Loc_max"]:
    omi_clean[col] = pd.to_numeric(omi_clean[col].astype("string").str.replace(",", ".", regex=False), errors="coerce")

# Derive a midpoint rent to simplify comparisons across zones and years.
omi_clean["rent_mid"] = (omi_clean["Loc_min"] + omi_clean["Loc_max"]) / 2

# Save the cleaned OMI panel for later use.
omi_clean_path = CLEAN_DIR / "omi_milan_cleaned.csv"
omi_clean.to_csv(omi_clean_path, index=False)

# Build a yearly citywide summary to use in the overview charts.
omi_yearly = omi_clean.groupby("Year", as_index=False).agg(loc_min_avg=("Loc_min", "mean"), loc_max_avg=("Loc_max", "mean"), rent_mid=("rent_mid", "mean"), zones=("Zona", "nunique"))
omi_yearly_path = CLEAN_DIR / "omi_milan_yearly_summary.csv"
omi_yearly.to_csv(omi_yearly_path, index=False)

# Display the merged structure and a small preview.
display(omi_yearly)
display(omi_clean.head(3))
print("Saved:", omi_clean_path)
print("Saved:", omi_yearly_path)

The OMI integration step transforms a fragmented archive into a real panel. This is analytically valuable because it preserves the official spatial vocabulary of the rental market while also creating a stable yearly series that can be compared with Airbnb activity. The midpoint rent is especially useful here, because it compresses the lower and upper quotation bounds into one interpretable reference without discarding the original fields.

The time span from 2018 to 2024 also complements the review history in a sensible way. OMI gives the institutional view of rent evolution, while reviews reveal platform activity. They do not measure the same phenomenon, but placing them side by side is precisely what allows the project to ask whether urban tourism pressure and formal rent dynamics move together or drift apart.

## 2) Mapping

The mapping step is necessary because the Airbnb and OMI datasets speak different spatial languages. Airbnb listings identify places through neighborhood names, while the OMI system organizes rents through official market zones called `Zona`. Without an explicit crosswalk, the two datasets can be described separately but not integrated in a way that supports neighborhood level interpretation.

The source of the mapping is the curated lookup table stored in `src/map_neighbourhoods_to_omi.py`. The notebook maps the `neighbourhood` field from the cleaned listings table to an `omi_zona` field, then uses that link to attach official rent levels to Airbnb snapshot metrics. This step is not a cosmetic merge. It is the bridge that allows the project to compare platform pressure with the formal geography of rents.

In [ ]:
# Load the curated lookup table from the project source folder.
spec = importlib.util.spec_from_file_location("map_neighbourhoods_to_omi", SRC_DIR / "map_neighbourhoods_to_omi.py")
mapping_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mapping_module)

# Apply the neighborhood to OMI zone mapping to the cleaned listings.
listings_mapped = listings_clean.copy()
listings_mapped["omi_zona"] = listings_mapped["neighbourhood"].map(mapping_module.NEIGHBOURHOOD_TO_ZONA)

# Measure how complete the spatial bridge is before merging anything.
mapping_match_rate = listings_mapped["omi_zona"].notna().mean()

# Build a neighborhood snapshot that summarizes current Airbnb structure.
neighbourhood_snapshot = listings_mapped.groupby(["neighbourhood", "omi_zona"], as_index=False).agg(
    listings_count=("id", "nunique"),
    median_price=("price", "median"),
    entire_home_share=("is_entire_home", "mean"),
    professional_host_share=("is_professional_host", "mean"),
    long_term_share=("is_long_term", "mean"),
    active_share=("is_active", "mean"),
)

# Extract the latest official rent context available from OMI.
omi_latest = omi_clean[omi_clean["Year"].eq(2024)].groupby("Zona", as_index=False).agg(omi_rent_2024=("rent_mid", "mean"), omi_loc_min_2024=("Loc_min", "mean"), omi_loc_max_2024=("Loc_max", "mean"))

# Join the neighborhood snapshot to the latest official rent context.
neighbourhood_snapshot = neighbourhood_snapshot.merge(omi_latest, left_on="omi_zona", right_on="Zona", how="left").drop(columns=["Zona"])

# Build a zone year context table for the longitudinal views.
zone_year_context = omi_clean.groupby(["Year", "Semester", "Zona"], as_index=False).agg(rent_mid=("rent_mid", "mean"), loc_min_avg=("Loc_min", "mean"), loc_max_avg=("Loc_max", "mean"))

# Save the mapped analytical tables for later notebooks and dashboards.
listings_mapped_path = CLEAN_DIR / "listings_milan_mapped.csv"
neighbourhood_snapshot_path = CLEAN_DIR / "neighbourhood_snapshot_with_omi_2024.csv"
zone_year_context_path = CLEAN_DIR / "omi_zone_year_context.csv"
listings_mapped.to_csv(listings_mapped_path, index=False)
neighbourhood_snapshot.to_csv(neighbourhood_snapshot_path, index=False)
zone_year_context.to_csv(zone_year_context_path, index=False)

# Display the mapping coverage and the first mapped neighborhoods.
print(f"Mapping match rate: {mapping_match_rate:.1%}")
display(neighbourhood_snapshot.sort_values("listings_count", ascending=False).head(10))
print("Saved:", listings_mapped_path)
print("Saved:", neighbourhood_snapshot_path)
print("Saved:", zone_year_context_path)

The reassuring result here is that the mapping covers the full set of Milan neighborhoods present in the listings snapshot. This means the curated crosswalk is not a fragile patch but a reliable integration layer for the city. That reliability matters because any missing zone assignment would have created a systematic blind spot precisely where the spatial argument of the project is strongest.

The mapped snapshot is also conceptually richer than the raw listings table. Once each neighborhood inherits an official rent context, we can ask more interesting questions about whether areas with more intense short term rental presence also sit in more expensive market zones. Even when that relationship is only descriptive, it already moves the project closer to an interpretable urban narrative.

## 3) EDA

The exploratory phase serves two complementary purposes. It documents the health of the data, and it identifies the patterns that are worth turning into final visual components. To keep the process reproducible, the notebook writes a textual report to `src/io/plos`, while also displaying compact profile tables inside the notebook so the reader can see the evidence directly.

In [ ]:
# Build a compact dictionary of analytical tables to profile.
eda_datasets = {
    "listings_clean": listings_clean,
    "reviews_clean": reviews_clean,
    "omi_clean": omi_clean,
    "neighbourhood_snapshot": neighbourhood_snapshot,
}

# Summarize rows, columns, and missingness for each dataset.
eda_overview = []
missing_frames = []
for name, frame in eda_datasets.items():
    eda_overview.append({"dataset": name, "rows": len(frame), "columns": frame.shape[1], "duplicate_rows": int(frame.duplicated().sum())})
    missing = frame.isna().sum().reset_index()
    missing.columns = ["column", "missing_values"]
    missing["dataset"] = name
    missing["missing_share"] = (missing["missing_values"] / len(frame)).round(4)
    missing_frames.append(missing.sort_values("missing_values", ascending=False).head(8))

# Combine the exploratory summaries into display tables.
eda_overview = pd.DataFrame(eda_overview)
eda_missing = pd.concat(missing_frames, ignore_index=True)

# Build cross dataset highlights for the textual report.
room_type_counts = listings_clean["room_type"].value_counts(dropna=False)
review_year_counts = reviews_clean["year"].value_counts().sort_index()
top_neighbourhoods = neighbourhood_snapshot.sort_values("listings_count", ascending=False).head(10)[["neighbourhood", "listings_count", "omi_rent_2024"]]
top_omi_zones = omi_clean["Zona"].value_counts().head(10)

# Write a reproducible text report for the submission folder.
report_lines = []
report_lines.append("Milan Airbnb and Rental Dynamics - EDA report")
report_lines.append("=" * 72)
report_lines.append("")
report_lines.append(f"Listings cleaned: {len(listings_clean):,} rows and {listings_clean.shape[1]} columns.")
report_lines.append(f"Reviews cleaned: {len(reviews_clean):,} rows and {reviews_clean.shape[1]} columns.")
report_lines.append(f"OMI cleaned: {len(omi_clean):,} rows and {omi_clean.shape[1]} columns.")
report_lines.append(f"Neighborhood snapshot: {len(neighbourhood_snapshot):,} rows and {neighbourhood_snapshot.shape[1]} columns.")
report_lines.append("")
report_lines.append("Listings highlights")
report_lines.append(f"Unique neighborhoods: {listings_clean['neighbourhood'].nunique()}")
report_lines.append(f"Entire home share: {listings_clean['is_entire_home'].mean():.1%}")
report_lines.append(f"Professional host share: {listings_clean['is_professional_host'].mean():.1%}")
report_lines.append(f"Missing price share: {listings_clean['price'].isna().mean():.1%}")
report_lines.append("")
report_lines.append("Reviews highlights")
report_lines.append(f"Unique reviewed listings: {reviews_clean['listing_id'].nunique():,}")
report_lines.append(f"Review window: {reviews_clean['date'].min().date()} to {reviews_clean['date'].max().date()}")
report_lines.append(f"Peak review year in the cleaned history: {int(review_year_counts.idxmax())}")
report_lines.append("")
report_lines.append("OMI highlights")
report_lines.append(f"Covered years: {omi_clean['Year'].min()} to {omi_clean['Year'].max()}")
report_lines.append(f"Unique OMI zones: {omi_clean['Zona'].nunique()}")
report_lines.append(f"Average midpoint rent in 2024: {omi_latest['omi_rent_2024'].mean():.2f}")
report_lines.append("")
report_lines.append("Top room types")
for label, value in room_type_counts.items():
    report_lines.append(f"{label}: {int(value):,}")
report_lines.append("")
report_lines.append("Top neighborhoods by listing count")
for row in top_neighbourhoods.itertuples(index=False):
    report_lines.append(f"{row.neighbourhood}: {int(row.listings_count):,} listings, OMI 2024 midpoint {row.omi_rent_2024:.2f}")
report_lines.append("")
report_lines.append("Most frequent OMI zones in the official panel")
for label, value in top_omi_zones.items():
    report_lines.append(f"{label}: {int(value):,} rows")

# Persist the EDA report inside the requested folder.
eda_report_path = PLOS_DIR / "eda_report.txt"
eda_report_path.write_text("\n".join(report_lines), encoding="utf-8")

# Export the tabular EDA summaries for convenience.
eda_overview.to_csv(PLOS_DIR / "eda_overview.csv", index=False)
eda_missing.to_csv(PLOS_DIR / "eda_missingness_top8.csv", index=False)

# Display the exploratory summaries and the beginning of the report.
display(eda_overview)
display(eda_missing.head(16))
print("\n".join(report_lines[:20]))
print("Saved:", eda_report_path)

The exploratory summaries show that the datasets are not equally informative, and that difference is itself an insight. The listings file is wide and behaviorally rich, but it is only a snapshot. The reviews file is narrower, yet it carries the strongest temporal signal. The OMI panel is smaller than both, but it is the only source that speaks directly about the formal rent market. The project becomes persuasive only when these strengths are combined instead of treated as interchangeable.

The EDA also reveals a strong spatial concentration. A limited set of neighborhoods accounts for a large share of the Airbnb stock, which suggests that the platform is not spread evenly across Milan. This concentration makes the mapping step more consequential, because a zone based rent measure becomes especially interesting when it is attached to neighborhoods that already stand out for intensity rather than for mere presence.

## 4) Overview - Visualization

The next figure is designed for a reader who may know Milan but not the datasets. The aim is to give one intuitive entry point into the project by combining market composition, neighborhood concentration, and the citywide temporal comparison between Airbnb review activity and official rent levels. This overview does not try to answer every analytical question. It creates the mental map needed to read the more pointed visualizations that follow.

In [ ]:
# Aggregate the review history to a yearly citywide series.
reviews_yearly = reviews_clean.groupby("year", as_index=False).agg(review_count=("listing_id", "size"), active_listings=("listing_id", "nunique"))

# Keep the overlapping years shared by reviews and OMI.
overview_yearly = reviews_yearly.merge(omi_yearly, left_on="year", right_on="Year", how="inner")
overview_yearly = overview_yearly[overview_yearly["year"].between(2018, 2024)].copy()

# Index the yearly signals to make them understandable on one axis.
overview_yearly["reviews_index"] = overview_yearly["review_count"] / overview_yearly["review_count"].iloc[0] * 100
overview_yearly["rent_index"] = overview_yearly["rent_mid"] / overview_yearly["rent_mid"].iloc[0] * 100

# Select the most relevant neighborhoods for the concentration panel.
top12 = neighbourhood_snapshot.sort_values("listings_count", ascending=False).head(12).sort_values("listings_count")

# Build the overview figure with three coordinated panels.
fig = plt.figure(figsize=(15, 10), facecolor=palette["white"])
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.1], width_ratios=[1, 1], hspace=0.28, wspace=0.18)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

# Plot the room type composition with the Milan palette.
room_mix = listings_clean["room_type"].value_counts().reset_index()
room_mix.columns = ["room_type", "count"]
sns.barplot(data=room_mix, x="count", y="room_type", ax=ax1, palette=[palette["coral"], palette["duomo_pink"], palette["gold"], palette["green"]])
ax1.set_title("Current Airbnb composition in Milan", fontsize=13, color=palette["red"])
ax1.set_xlabel("Listings")
ax1.set_ylabel("")

# Plot the concentration of listings across neighborhoods.
sns.barplot(data=top12, x="listings_count", y="neighbourhood", ax=ax2, color=palette["terracotta"])
ax2.set_title("Neighborhoods with the largest Airbnb stock", fontsize=13, color=palette["red"])
ax2.set_xlabel("Listings")
ax2.set_ylabel("")

# Plot the indexed temporal comparison on the full bottom row.
ax3.plot(overview_yearly["year"], overview_yearly["reviews_index"], color=palette["coral"], marker="o", linewidth=2.5, label="Airbnb review activity index")
ax3.plot(overview_yearly["year"], overview_yearly["rent_index"], color=palette["green"], marker="o", linewidth=2.5, label="OMI midpoint rent index")
ax3.axvspan(2020, 2020.9, color=palette["duomo_pink"], alpha=0.35)
ax3.text(2020.08, ax3.get_ylim()[1] * 0.96, "Pandemic shock", color=palette["red"], fontsize=10)
ax3.set_title("Citywide trend comparison, 2018 = 100", fontsize=13, color=palette["red"])
ax3.set_xlabel("Year")
ax3.set_ylabel("Indexed value")
ax3.legend(frameon=False, loc="upper left")

# Add a narrative title and export the figure.
fig.suptitle("Milan in one glance: supply structure, concentration, and trend context", fontsize=16, color=palette["red"], y=0.98)
overview_plot_path = PLOS_DIR / "overview_visualization.png"
fig.savefig(overview_plot_path, bbox_inches="tight")
plt.show()
print("Saved:", overview_plot_path)


The overview makes three patterns legible at once. First, entire homes dominate the current Airbnb offer, which strengthens the idea that the platform is closely tied to the use of full residential units rather than only spare rooms. Second, the stock is strongly concentrated in a subset of neighborhoods, which means any discussion of impact should be spatial rather than purely citywide. Third, review activity and official rents both recover after the pandemic, although they do so with different intensity, suggesting that the two systems are related but not mechanically identical.

This is the kind of summary a non expert reader needs. It avoids technical overload, but it still conveys that the project is about structure, concentration, and temporal context, not about a simplistic one variable explanation of urban change.

## 5) Black Hat Visualization

A black hat view should use the same data but manipulate perception through design choices. The aim is not to produce false numbers, but to show how a chart can become misleading when it hides uncertainty, truncates context, and invites an emotional conclusion that the evidence does not actually justify.

In [ ]:
# Reuse the overlapping yearly series but keep only the recent recovery.
black_hat = overview_yearly[overview_yearly["year"].between(2021, 2024)].copy()

# Create the dual axis chart with aggressive styling choices.
fig, ax1 = plt.subplots(figsize=(12, 6), facecolor="#111111")
ax1.set_facecolor("#111111")
ax2 = ax1.twinx()

# Plot Airbnb activity with a dramatic color and thick stroke.
ax1.plot(black_hat["year"], black_hat["review_count"], color=palette["coral"], marker="o", linewidth=4, markersize=8)
ax1.fill_between(black_hat["year"], black_hat["review_count"], color=palette["coral"], alpha=0.18)

# Plot rent on a tightly truncated scale to exaggerate movement.
ax2.plot(black_hat["year"], black_hat["rent_mid"], color=palette["gold"], marker="o", linewidth=4, markersize=8)
ax2.set_ylim(black_hat["rent_mid"].min() - 0.25, black_hat["rent_mid"].max() + 0.15)

# Style the axes to intensify the impression of a causal surge.
ax1.set_title("Airbnb explodes and Milan rents follow immediately", fontsize=17, color=palette["white"], pad=16)
ax1.set_xlabel("Year", color=palette["white"])
ax1.set_ylabel("Airbnb reviews", color=palette["coral"])
ax2.set_ylabel("OMI midpoint rent", color=palette["gold"])
ax1.tick_params(colors=palette["white"])
ax2.tick_params(colors=palette["white"])
for spine in ax1.spines.values():
    spine.set_color("#666666")
for spine in ax2.spines.values():
    spine.set_color("#666666")

# Add a suggestive annotation that overstates the visual evidence.
ax1.text(2021.1, black_hat["review_count"].max() * 0.93, "Looks like one market is dragging the other", color=palette["duomo_pink"], fontsize=11)

# Export the intentionally misleading figure.
black_hat_path = PLOS_DIR / "black_hat_visualization.png"
fig.savefig(black_hat_path, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Saved:", black_hat_path)


This figure is persuasive precisely because it is irresponsible. By dropping the pre 2021 context, it erases the pandemic break and makes the rebound look like a seamless acceleration. By using dual axes and a truncated rent scale, it visually amplifies alignment between two variables that live on completely different magnitudes. The title then does the final manipulative step, because it turns a visual coincidence into a causal slogan.

The lesson is useful for the project report. Ethical problems in visualization often come from emphasis and framing rather than fabricated data. A reader can be pushed toward a strong conclusion even when every point on the chart is technically real.

## 6) White Hat Visualization

The white hat view keeps the same underlying comparison but redesigns it to be transparent. Instead of relying on two vertical scales, it normalizes both series to a common baseline and restores the full overlapping period. This approach gives up some dramatic effect, but it gives the reader a fair chance to judge how similar or dissimilar the trajectories really are.

In [ ]:
# Copy the overlap table and keep the full shared time window.
white_hat = overview_yearly.copy()

# Plot both series on the same indexed scale for an honest comparison.
fig, ax = plt.subplots(figsize=(12, 6), facecolor=palette["white"])
ax.plot(white_hat["year"], white_hat["reviews_index"], color=palette["coral"], marker="o", linewidth=2.8, label="Airbnb review activity index")
ax.plot(white_hat["year"], white_hat["rent_index"], color=palette["green"], marker="o", linewidth=2.8, label="OMI midpoint rent index")

# Keep the context of the pandemic visible without turning it into drama.
ax.axvspan(2020, 2020.9, color=palette["duomo_pink"], alpha=0.25)
ax.text(2020.05, ax.get_ylim()[1] * 0.97, "Pandemic shock", color=palette["red"], fontsize=10)

# Use direct labels and a neutral title to reduce interpretive pressure.
ax.set_title("Airbnb activity and official rents recover together, but not at the same pace", fontsize=15, color=palette["red"], pad=14)
ax.set_xlabel("Year")
ax.set_ylabel("Indexed value, 2018 = 100")
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.25)

# Add a concise note that makes the design choice explicit.
fig.text(0.12, 0.02, "Both lines are indexed to 2018 so that movement is comparable without dual axes.", color=palette["terracotta"], fontsize=10)

# Export the transparent figure to the project io folder.
white_hat_path = PLOS_DIR / "white_hat_visualization.png"
fig.savefig(white_hat_path, bbox_inches="tight")
plt.show()
print("Saved:", white_hat_path)


The white hat version still supports a substantive interpretation, but it does so more carefully. The reader can see that Airbnb review activity rebounds more sharply than official rents, and that the two lines become closer again after the pandemic disruption. What the chart does not do is overpromise. It does not claim that the rent series proves platform causality, only that the two dynamics deserve to be studied together.

This difference between the black hat and white hat designs is valuable beyond the chart itself. It clarifies the ethical standard of the whole project. Good visualization is not only about making patterns visible. It is about choosing forms that do not pressure the reader into conclusions that the data cannot fully sustain.